In [33]:
import requests
import streamlit as st

# ---- Settings ----------------------------------------------------------
BUSINESS = "An Outpatient Healthcare Facility"
LABELS = ["Scheduling", "Wait Times", "Staff Experience", "Positive Experience", "Needs Review"]
GUIDE = """Classify each patient comment using exactly one of these labels:

Scheduling: Issues with appointment booking, availability, cancellations, or rescheduling.
Wait Times: Complaints about delays before or during appointments.
Staff Experience: Feedback about staff communication, professionalism, or helpfulness.
Positive Experience: General praise or satisfaction without a specific complaint.
Needs Review: Serious concerns, unresolved complaints, or unclear feedback requiring management attention.

If a comment mentions multiple issues, choose the label representing the main concern.
Prioritize needs review when the comment describes a serious issue.
Return only the label without any additional explanation."""

MODEL = "openai/gpt-oss-20b"   # if you get a 404, the model was retired: see console.groq.com/docs/models
URL = "https://api.groq.com/openai/v1/chat/completions"
# ------------------------------------------------------------------------

def label_comment(text):
    key = st.secrets["GROQ_API_KEY"]
    prompt = f"You label customer comments for {BUSINESS}.\n{GUIDE}\n\nComment: {text}"
    response = requests.post(
        URL,
        headers={"Authorization": f"Bearer {key}"},
        json={"model": MODEL, "temperature": 0,
              "messages": [{"role": "user", "content": prompt}]},
        timeout=30,
    )
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]
    label = answer.strip().title()
    return label if label in LABELS else "Needs Review"